# Cell-type based metrics
1. Cell type distribution (all / vascular cell types)
2. Characterising Undefined cells
3. Clustering quality (CH/SH score) and Harmony UMAPs

NOTE: CH/SH scores computed per sample, as batch effects otherwise impact computation.

In [ ]:
cohort = "aging"  # aging, VizgenMouseBrain, ABCAtlas

## 1. Cell type distribution

In [ ]:
from cellseg_benchmark.metrics import compute_cell_type_distribution, compute_metric_for_all_methods, plot_cell_type_distribution
for adata_name, ct in [("adata_integrated", "cell_type_revised"), ("adata_vascular_subset", "cell_type_incl_zonation")]:
    compute_metric_for_all_methods(
        compute_cell_type_distribution,
        results_name=f"cell_type_metrics/cell_type_distribution_{adata_name}_{ct}.csv",
        cohort=cohort,
        celltype_name=ct,
        adata_name=adata_name,
        overwrite=False,
    )

In [ ]:
plot_cell_type_distribution(cohort, "adata_integrated_cell_type_revised", show=True)
plot_cell_type_distribution(cohort, "adata_vascular_subset_cell_type_incl_zonation", show=True)

## 2. Characterising Undefined cells
Cells labelled `Undefined` in `cell_type_revised` after `merge_adata` QC (counts, genes, volume, spatial outliers), i.e. on `adata_integrated`. `compute_annotation_qc` reruns `annotation_qc_summary` on these cells only.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from cellseg_benchmark import BASE_PATH
from cellseg_benchmark._constants import method_colors
from cellseg_benchmark.metrics import clean_method_name, compute_annotation_qc, compute_metric_for_all_methods
compute_metric_for_all_methods(
    compute_annotation_qc,
    results_name="cell_type_metrics/annotation_qc_adata_integrated.csv",
    cohort=cohort,
    overwrite=False,
)
res_dir = Path(BASE_PATH) / "metrics" / cohort / "cell_type_metrics"
plot_path = res_dir / "plots"
plot_path.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(res_dir / "annotation_qc_adata_integrated.csv", index_col=0)
pooled = df[df["sample"] == "all"].set_index("method")
per_sample = df[df["sample"] != "all"]
order = pooled["pct_undefined"].sort_values().index
labels = [clean_method_name(m) for m in order]
colors = [method_colors.get(m, "grey") for m in order]

### % Undefined per method

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=order, y=pooled.loc[order, "pct_undefined"], palette=colors, ax=ax)
sns.stripplot(data=per_sample, x="method", y="pct_undefined", order=order, color="k", size=3, ax=ax)
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set(xlabel="", ylabel="% Undefined (post-QC)")
sns.despine()
plt.savefig(plot_path / "pct_undefined.png", dpi=300, bbox_inches="tight")
plt.show()

### Pre- vs post-QC
Pre-QC: `annotation_qc.csv` written by `cell_type_annotation.py` per sample.

In [ ]:
pre = pd.concat(pd.read_csv(f) for f in Path(BASE_PATH).glob("samples/*/results/*/cell_type_annotation/annotation_qc.csv"))
pre = pre[pre["sample"].isin(per_sample["sample"])]
pre_post = (
    pre.groupby("seg_method")
    .apply(lambda g: (g.pct_undefined * g.n_cells).sum() / g.n_cells.sum())
    .rename("pre")
    .to_frame()
    .join(pooled["pct_undefined"].rename("post"))
    .dropna()
)
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(pre_post.pre, pre_post.post, c=[method_colors.get(m, "grey") for m in pre_post.index])
lim = [0, pre_post.max().max() * 1.05]
ax.plot(lim, lim, "k--", lw=0.8)
ax.set(xlabel="% Undefined pre-QC", ylabel="% Undefined post-QC", xlim=lim, ylim=lim)
sns.despine()
plt.show()
pre_post.assign(diff=pre_post.post - pre_post.pre).sort_values("diff").round(2)

### Undefined per pipeline step
SCALPEL QC failed (per cell) -> cluster vote -> marker revision (final).

In [ ]:
steps = {"pct_scalpel_failed": "SCALPEL failed", "pct_undefined_vote": "after vote", "pct_undefined": "final"}
long = pooled.loc[order, list(steps)].rename(columns=steps).set_axis(labels).reset_index(names="method")
long = long.melt(id_vars="method", var_name="step", value_name="pct")
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=long, x="method", y="pct", hue="step", ax=ax)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.set(xlabel="", ylabel="% cells")
sns.despine()
plt.show()

### What are Undefined cells?
`undefined_mixed_excess`: MapMyCells mixed rate of Undefined minus assigned cells at matched counts (pp).
~0: low quality; > 0: mixed identity (e.g. segmentation merging cells) beyond their counts.

In [ ]:
vals = pooled.assign(
    counts_ratio=pooled.median_counts_undefined / pooled.median_counts_assigned,
    cor_diff=pooled.median_mmc_cor_undefined - pooled.median_mmc_cor_assigned,
).loc[order]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, col, ref, xlabel in [
    (axes[0], "undefined_mixed_excess", 0, "Undefined mixed excess (pp)"),
    (axes[1], "counts_ratio", 1, "median counts, Undefined / assigned"),
    (axes[2], "cor_diff", 0, "median MMC cor, Undefined - assigned"),
]:
    ax.barh(labels, vals[col], color=colors)
    ax.axvline(ref, color="k", lw=0.8)
    ax.set_xlabel(xlabel)
sns.despine()
plt.tight_layout()
plt.show()

## 3. Clustering scores
CH/SH on annotated cell types in PCA space (30 PCs), per sample. Use `celltype_name="leiden"` for intrinsic clusterability instead.

In [ ]:
import math
from pathlib import Path
import anndata as ad
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cellseg_benchmark import BASE_PATH
from cellseg_benchmark._constants import cell_type_colors, method_colors, method_names
from cellseg_benchmark.metrics import compute_clustering_scores, compute_metric_for_all_methods, plot_clustering_scores
for adata_name, ct in [("adata_integrated", "cell_type_revised"), ("adata_vascular_subset", "cell_type_incl_zonation")]:
    compute_metric_for_all_methods(
        compute_clustering_scores,
        results_name=f"cell_type_metrics/clustering_score_{adata_name}_{ct}.csv",
        cohort=cohort,
        celltype_name=ct,
        adata_name=adata_name,
        overwrite=False,
    )

In [ ]:
plot_clustering_scores(cohort, "adata_integrated_cell_type_revised", show=True)
plot_clustering_scores(cohort, "adata_vascular_subset_cell_type_incl_zonation", show=True)

### Harmony UMAPs per method

In [ ]:
analysis = Path(BASE_PATH) / "analysis" / cohort
grid = [m for m in method_colors if (analysis / m / "adatas" / "adata_integrated.h5ad.gz").exists()]
ncol, s, ws, hs = 6, 3, 0, 0.2
nrow = math.ceil(len(grid) / ncol)
fig, axs = plt.subplots(
    nrow,
    ncol,
    figsize=(s * (ncol + ws * (ncol - 1)), s * (nrow + hs * (nrow - 1))),
    gridspec_kw=dict(left=0, right=1, bottom=0, top=1, wspace=ws, hspace=hs),
)
for ax in axs.flat:
    ax.axis("off")
for i, (ax, m) in enumerate(zip(axs.flat, grid)):
    print(f"{i + 1}/{len(grid)} {m}")
    with h5py.File(analysis / m / "adatas" / "adata_integrated.h5ad.gz") as f:
        xy = f["obsm/X_umap_harmony_20_50"][:]
        ct = pd.Series(ad.io.read_elem(f["obs/cell_type_revised"])).astype(str)
    idx = np.random.default_rng(0).permutation(len(ct))[:200_000]
    ax.scatter(*xy[idx].T, c=ct.iloc[idx].map(cell_type_colors).fillna("#D9D9D9"), s=0.3, lw=0, rasterized=True)
    ax.set_title(method_names.get(m, m), fontsize=17, pad=3)
    ax.margins(0.01)
handles = [plt.Line2D([], [], marker="o", ls="", color=v, label=k) for k, v in cell_type_colors.items() if k in ct.unique()]
fig.legend(handles=handles, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=10)
fig.savefig(Path(BASE_PATH) / "metrics" / cohort / "cell_type_metrics" / "plots" / "umap_harmony_grid.png", dpi=300, bbox_inches="tight")
plt.show()